<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/tapta-data-scalability/notebooks/tapta_multiseed_final_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 TAPTA: Multi-Seed Verification (5 Seeds: 2021 – 2025)
### Đề tài: Continual Relation Extraction with Task-Aware Prompt Adaptation (Thạc sĩ)

Notebook này dùng để nghiệm thu số liệu thống kê chuẩn khoa học (Mean $\pm$ Std) cho **phương pháp đề xuất chính thức của luận văn** trên cả hai benchmark lớn:
- **FewRel Track A:** 80 quan hệ, 8 tác vụ liên tiếp, strict 5-shot, $M=0$ Rehearsal-Free.
- **TACRED Track A:** 41 quan hệ, 8 tác vụ liên tiếp, strict 5-shot, $M=0$ Rehearsal-Free.

> **Ràng buộc nghiên cứu:** Strict Equal 5-shot (cả Task 1), $M=0$ (không lưu câu thô hay dùng LLM sinh mẫu), 100% Frozen BERT-base 109.5M, Task-Agnostic Class-Incremental (CIL).

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [ ]:
import torch
print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU.")
print("=" * 60)

## 2. Clone / Cập Nhật Repository `research-code-lab`

In [ ]:
import os
REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
BRANCH = "feat/tapta-data-scalability"
REPO_DIR = "/content/research-code-lab"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scikit-learn tabulate pytest

## 3. Chạy Multi-Seed Verification Trên TACRED Track A
*(41 quan hệ, 8 tác vụ, 5 seeds: 2021, 2022, 2023, 2024, 2025 ~ 12 phút)*

In [ ]:
import os
SEEDS = [2021, 2022, 2023, 2024, 2025]

for seed in SEEDS:
    print("\n" + "=" * 75)
    print(f">>> CHẠY TACRED TRACK A VỚI SEED {seed}...")
    print("=" * 75)
    cmd = f"python experiments/run_p5.py --config configs/p5_candidates/p5_tacred_5shot_seed2021.yaml --mode p4_baseline --seed {seed} --device cuda"
    ret = os.system(cmd)
    if ret != 0:
        print(f"⚠️ CẢNH BÁO: Lệnh chạy seed {seed} trả về mã lỗi: {ret}")


## 4. Chạy Multi-Seed Verification Trên Benchmark Chính FewRel Track A
*(80 quan hệ, 8 tác vụ, 5 seeds: 2021, 2022, 2023, 2024, 2025 ~ 30 phút)*

In [ ]:
import os
SEEDS = [2021, 2022, 2023, 2024, 2025]

for seed in SEEDS:
    print("\n" + "=" * 75)
    print(f">>> CHẠY FEWREL TRACK A VỚI SEED {seed}...")
    print("=" * 75)
    cmd = f"python experiments/run_p5.py --config configs/p5_candidates/p5_fewrel_5shot_seed2021.yaml --mode p4_baseline --seed {seed} --device cuda"
    ret = os.system(cmd)
    if ret != 0:
        print(f"⚠️ CẢNH BÁO: Lệnh chạy seed {seed} trả về mã lỗi: {ret}")


## 5. Tổng Hợp Thống Kê Chuẩn Khoa Học (Mean $\pm$ Std) & Xuất Mã LaTeX
Đọc toàn bộ kết quả của 5 seeds, tính giá trị trung bình, độ lệch chuẩn và tạo bảng LaTeX sẵn sàng chèn vào Luận văn.

In [ ]:
!python scripts/aggregate_multiseed_results.py --runs_dir /content/research-code-lab/runs --mode all --output /content/research-code-lab/runs/multiseed_summary.json


## 6. Đóng Gói Tự Động & Tải Về Toàn Bộ Kết Quả (Artifacts Bundle)

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import files

print("Đang đóng gói kết quả thực nghiệm 5 seeds...")
export_dir = Path("/content/multiseed_bundle")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

runs_dir = Path("/content/research-code-lab/runs")
for run_folder in runs_dir.glob("*"):
    if run_folder.is_dir():
        dest = export_dir / run_folder.name
        shutil.copytree(run_folder, dest, ignore=shutil.ignore_patterns("*.pt"))

zip_path = "/content/tapta_multiseed_final_results"
shutil.make_archive(zip_path, "zip", export_dir)
zip_file = zip_path + ".zip"
file_size_kb = os.path.getsize(zip_file) / 1024
print(f"Tệp nén đã tạo: {zip_file} (Kích thước: {file_size_kb:.1f} KB)")
print("Tự động kích hoạt tải xuống...")
files.download(zip_file)
